# Transformación y preparación de datos

En este notebook se crea una copia del catálogo de USGS para los análisis posteriores. El archivo de `data/raw/` se conserva sin modificaciones: las transformaciones se realizan sobre `df` en memoria y al final se guarda una copia en `data/processed/`.

# 1. Importaciones y rutas


In [48]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import Polygon


def find_project_root():
    """Busca la carpeta que contiene el CSV crudo."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "sismos_usgs.csv").exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "sismos_usgs.csv"
GEO_PATH = PROJECT_ROOT / "data" / "raw" / "geodata"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH = PROCESSED_DIR / "sismos_argentina_limpio.csv"

# 2. Carga y copia de trabajo

Primero se carga el archivo original y se crea una copia. De esta forma, `df_raw` sirve como referencia y todas las transformaciones se realizan sobre `df`.

In [49]:
df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()

required_columns = {
    "id", "time", "updated", "latitude", "longitude", "depth", "mag",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {sorted(missing_columns)}")

print(f"Filas iniciales: {df.shape[0]:,}")
print(f"Columnas iniciales: {df.shape[1]}")
print(f"El archivo crudo conserva {df_raw.shape[0]:,} filas")

Filas iniciales: 10,795
Columnas iniciales: 22
El archivo crudo conserva 10,795 filas


# 3. Revisión inicial de calidad

Antes de eliminar o imputar datos se observa la cantidad de nulos, duplicados y los tipos de cada columna.

In [50]:
print("Tipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum().sort_values(ascending=False))

print(f"\nDuplicados completos: {df.duplicated().sum()}")
print(f"IDs duplicados: {df['id'].duplicated().sum()}")

Tipos de datos:
time                   str
latitude           float64
longitude          float64
depth              float64
mag                float64
magType                str
nst                float64
gap                float64
dmin               float64
rms                float64
net                    str
id                     str
updated                str
place                  str
type                   str
horizontalError    float64
depthError         float64
magError           float64
magNst             float64
status                 str
locationSource         str
magSource              str
dtype: object

Valores nulos por columna:
nst                6470
magNst              536
magError            534
gap                  56
dmin                 56
horizontalError       2
rms                   2
mag                   0
magType               0
longitude             0
latitude              0
time                  0
depth                 0
net                   0
place       

# 4. Selección geográfica de Argentina

Se repite el criterio espacial del notebook 01: un evento pertenece al área de estudio si su punto se encuentra dentro de las provincias, la plataforma continental o el polígono antártico utilizado para el proyecto. El filtro se realiza sobre una copia y no cambia el CSV original.

In [ ]:
gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",
)

provincias = gpd.read_file(GEO_PATH / "provinciaPolygon.shp")
plataforma = gpd.read_file(GEO_PATH / "plataforma_continentalPolygon.shp")

coordenadas_antartida = [
    (-74.0, -60.0), (-25.0, -60.0), (-25.0, -90.0),
    (-74.0, -90.0), (-74.0, -60.0),
]
poligono_antartico = Polygon(coordenadas_antartida)
poligono_nacional = (
    provincias.geometry.union_all()
    .union(plataforma.geometry.union_all())
    .union(poligono_antartico)
)

df = pd.DataFrame(
    gdf[gdf.geometry.within(poligono_nacional)].drop(columns="geometry")
)

df1 = df.copy()  # Se usa para comparar antes/después

print(f"Filas dentro del área de estudio: {df.shape[0]:,}")

Filas dentro del área de estudio: 6,431


# 5. Eliminación de columnas

En el EDA se identificaron seis columnas que no aportan información útil para el análisis. Se eliminan por dos motivos distintos.

**Por exceso de valores faltantes:**

- `nst` (cantidad de estaciones sísmicas utilizadas para localizar el evento): presenta valores faltantes en el 60,2 % de los registros (3.873 de 6.431). Debido a esta elevada proporción de faltantes, imputar la variable implicaría estimar una gran cantidad de valores, mientras que eliminar los registros con datos faltantes supondría perder más de la mitad del conjunto de datos. Por estos motivos, se decide eliminar la variable.

**Por falta de variabilidad:** en estas columnas, una sola categoría concentra casi todos los registros. Cuando una variable presenta prácticamente el mismo valor en todas las filas, aporta poca información para diferenciar los eventos entre sí y, por lo tanto, no resulta útil para el análisis.

- `net` (red sísmica que aportó la solución preferida): 6.427 de 6.429 registros son `us`. Los dos restantes son los registros `iscgem` que se eliminan en la sección siguiente.
- `type` (tipo de evento): el 100 % de los registros son `earthquake`, no se identifican otros tipos de eventos en el dataset.
- `status` (estado de revisión): el 100 % está `reviewed`, es decir, todos los eventos fueron revisados por un sismólogo.
- `locationSource` (red que calculó la ubicación): el 99,9 % es `us`.
- `magSource` (red que calculó la magnitud): el 96,5 % es `us`.

La columna `magType` también es categórica, pero se conserva porque aporta información sobre el método utilizado para determinar la magnitud de cada evento (`mb`, `mww`, `mwr`, `ml`).

In [52]:
columns_to_drop = [
    "nst", "net", "type", "status", "locationSource", "magSource",
]
existing_columns_to_drop = [
    column for column in columns_to_drop if column in df.columns
]

columnas_antes = df.shape[1]
df = df.drop(columns=existing_columns_to_drop)

pd.DataFrame({
    "Antes": [columnas_antes],
    "Después": [df.shape[1]],
    "Columnas eliminadas": [", ".join(existing_columns_to_drop)],
}, index=["Cantidad de columnas"])

,Antes,Después,Columnas eliminadas
Cantidad de columnas,22,16,"nst, net, type, status, locationSource, magSource"


# 6. Eliminación de registros incompletos

En el EDA se identificaron cuatro eventos a los que les faltan casi todas las variables de calidad (`nst`, `gap`, `dmin`, `rms`, `horizontalError`, `magNst`). Estas variables aportan información sobre la calidad de las mediciones, por lo que su ausencia limita la posibilidad de evaluar la calidad de estos registros.

- `iscgem621613005` e `iscgem620210242`: además de los faltantes, tienen magnitudes atípicas (5,06 y 5,18) respecto del resto del catálogo.
- `us10007f0t` y `us100073ln`: tienen 5 o más variables de calidad faltantes.

El resto de los registros con faltantes tiene como máximo 3 valores nulos y se conservan porque estos valores se pueden completar mediante imputación (ver sección 8).

Los registros se eliminan por su `id`, que es un identificador estable, y no por su número de fila, que cambia cada vez que se filtran los datos.

In [53]:
suspicious_ids = {"iscgem621613005", "iscgem620210242","us10007f0t", "us100073ln"}
present_suspicious_ids = set(df.loc[df["id"].isin(suspicious_ids), "id"])

registros_eliminados = df[df["id"].isin(present_suspicious_ids)]
filas_antes = len(df)

df = df.loc[~df["id"].isin(present_suspicious_ids)].copy()

display(registros_eliminados)

pd.DataFrame(
    {"Filas": [filas_antes, len(df)]},
    index=["Antes", "Después"],
)

,time,latitude,longitude,depth,mag,magType,gap,dmin,rms,id,updated,place,horizontalError,depthError,magError,magNst
4848,2021-12-04T04:32:02.340Z,-59.762,-29.666,10.0,5.06,mw,NaN,NaN,NaN,iscgem621613005,2025-12-22T22:21:35.177Z,South Sandwich Islands region,NaN,25.0,0.1,NaN
6306,2021-03-14T14:09:13.950Z,-59.303,-31.041,10.0,5.18,mw,NaN,NaN,NaN,iscgem620210242,2025-12-22T18:55:19.233Z,Scotia Sea,NaN,25.0,0.1,NaN
10522,2016-12-05T18:06:04.000Z,-23.185,-66.656,223.0,5.00,mww,NaN,NaN,1.88,us10007f0t,2025-12-22T17:19:40.859Z,"99 km W of El Aguilar, Argentina",7.1,6.0,NaN,NaN
10628,2016-11-02T01:26:00.000Z,-31.498,-64.375,33.0,3.30,md,NaN,NaN,0.59,us100073ln,2017-01-24T02:02:14.040Z,"4 km SSW of Malagueño, Argentina",11.2,10.1,NaN,NaN


,Filas
Antes,6431
Después,6427


# 7. Fechas y variables temporales

Las columnas `time` (momento en que ocurrió el sismo) y `updated` (última actualización del registro) vienen como texto. Mientras sean texto no se pueden ordenar cronológicamente, calcular diferencias de tiempo ni agrupar por período. Por eso se convierten a tipo fecha, en horario UTC, que es el que usa USGS.

A partir de `time` se crean cuatro variables nuevas:

- `event_year`: año del evento. Permite analizar la evolución de la actividad sísmica a lo largo de los años.
- `event_month`: mes del evento. Permite detectar posibles patrones estacionales.
- `event_day`: día del mes.
- `event_hour`: hora del evento (UTC).

Estas variables se van a usar en el análisis temporal y en el modelo supervisado, que necesita separar los datos por períodos.

In [54]:
for column in ["time", "updated"]:
    df[column] = pd.to_datetime(df[column], utc=True, errors="raise")

df["event_year"]  = df["time"].dt.year
df["event_month"] = df["time"].dt.month
df["event_day"]   = df["time"].dt.day
df["event_hour"]  = df["time"].dt.hour

print(df[["time", "updated", "event_year", "event_month", "event_day", "event_hour"]].head())

                              time                          updated  \
0 2026-09-14 05:03:12.877000+00:00 2026-09-14 06:21:54.040000+00:00   
1 2026-09-13 19:40:25.496000+00:00 2026-09-24 17:19:43.040000+00:00   
3 2026-09-12 11:05:46.433000+00:00 2026-09-23 07:20:17.040000+00:00   
4 2026-09-09 10:06:57.170000+00:00 2026-09-09 10:38:25.040000+00:00   
7 2026-09-08 06:50:46.470000+00:00 2026-09-08 07:06:29.040000+00:00   

   event_year  event_month  event_day  event_hour  
0        2026            9         14           5  
1        2026            9         13          19  
3        2026            9         12          11  
4        2026            9          9          10  
7        2026            9          8           6  


# 8. Tratamiento de valores faltantes

Se imputan los valores faltantes de las variables de calidad que se conservaron, ya que presentan una baja proporción de valores faltantes respecto del total:

| Variable | Qué mide | % faltante |
|---|---|---|
| `gap` | Brecha azimutal entre estaciones (grados) | 0,12 % |
| `dmin` | Distancia a la estación más cercana (grados) | 0,12 % |
| `rms` | Error de ajuste de los tiempos de llegada (segundos) | 0,03 % |
| `horizontalError` | Incertidumbre de la ubicación horizontal (km) | 0,03 % |
| `magError` | Incertidumbre de la magnitud | 3,5 % |
| `magNst` | Estaciones usadas para calcular la magnitud | 3,6 % |

Se utiliza la mediana y no la media porque estas variables presentan valores extremos, como se observó en los boxplots del EDA. La media puede verse afectada por estos valores, mientras que la mediana representa mejor el valor central de la distribución.

`depthError` no necesita imputación porque no presenta valores faltantes, y `nst` ya fue eliminada en la sección 5.

In [55]:

numeric_imputation_columns = [
    "gap", "dmin", "rms", "horizontalError", "magError", "magNst",
]
numeric_imputation_columns = [
    column for column in numeric_imputation_columns if column in df.columns
]

nulos_antes = df[numeric_imputation_columns].isnull().sum()
medianas = {}

for column in numeric_imputation_columns:
    df[column] = pd.to_numeric(df[column], errors="raise")
    median = df[column].median()
    if pd.isna(median):
        raise ValueError(f"No se puede imputar {column}: mediana nula")
    medianas[column] = median
    df[column] = df[column].fillna(median)


pd.DataFrame({
    "Nulos antes": nulos_antes,
    "Mediana utilizada": pd.Series(medianas).round(3),
    "Nulos después": df[numeric_imputation_columns].isnull().sum(),
})

,Nulos antes,Mediana utilizada,Nulos después
gap,4,79.000,0
dmin,4,4.844,0
rms,0,0.730,0
horizontalError,0,9.420,0
magError,225,0.123,0
magNst,225,18.000,0


Notar que las variables "rms" y "horizontalError" ya no presentan valores nulos, debido a que los registros que contenían estos faltantes fueron eliminados en la sección 6.

# 9. Comprobaciones finales

Se verifica automáticamente que el conjunto cumpla:

- cada `id` aparece una sola vez y no hay filas duplicadas;
- `time` y `updated` son de tipo fecha;
- no quedan faltantes en las variables principales ni en las imputadas.

Si alguna condición no se cumple, el notebook se detiene con un error. Después se compara el catálogo original con el resultado final.

In [ ]:
essential_columns = ["latitude", "longitude", "depth", "mag"]


assert not df["id"].duplicated().any()
assert not df.duplicated().any()
assert isinstance(df["time"].dtype, pd.DatetimeTZDtype)
assert isinstance(df["updated"].dtype, pd.DatetimeTZDtype)
assert df[essential_columns].notna().all().all()
assert df[numeric_imputation_columns].notna().all().all()

resumen = pd.DataFrame({
    "Antes (archivo crudo)": [
        df_raw.shape[0], df_raw.shape[1],
        df_raw.duplicated().sum(), int(df_raw.isna().sum().sum()),
    ],
    "Antes (archivo Argentina)": [
        df1.shape[0], df1.shape[1],
        df1.duplicated().sum(), int(df1.isna().sum().sum()),
    ],
    "Después (transformado)": [
        df.shape[0], df.shape[1],
        df.duplicated().sum(), int(df.isna().sum().sum()),
    ],
}, index=["Filas", "Columnas", "Filas duplicadas", "Valores nulos"])

display(resumen)
display(df.describe(include="all").transpose())

,Antes (archivo crudo),Antes (archivo Argentina),Después (transformado)
Filas,10795,6431,6427
Columnas,22,22,20
Filas duplicadas,0,0,0
Valores nulos,7656,4349,0


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
time,6427,NaN,NaN,NaN,2021-10-29 10:44:38.234216+00:00,2016-09-15 01:06:30.870000+00:00,2020-01-30 22:11:51.945500+00:00,2021-09-07 01:52:54.733000+00:00,2023-10-21 06:03:08.061500+00:00,2026-09-14 05:03:12.877000+00:00,NaN
latitude,6427.0,NaN,NaN,NaN,-45.515626,-65.3437,-58.75835,-55.8503,-24.9253,-21.8795,15.799365
longitude,6427.0,NaN,NaN,NaN,-45.236885,-73.2752,-66.93575,-30.8068,-26.05445,-25.0017,19.741006
depth,6427.0,NaN,NaN,NaN,90.128827,1.0,10.0,43.33,167.5765,626.01,94.031457
mag,6427.0,NaN,NaN,NaN,4.588548,2.3,4.3,4.5,4.8,8.1,0.395059
magType,6427,7,mb,5650,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gap,6427.0,NaN,NaN,NaN,86.644469,12.0,58.0,79.0,107.0,292.0,40.155364
dmin,6427.0,NaN,NaN,NaN,5.38621,0.041,1.615,4.844,7.453,32.456,4.687069
rms,6427.0,NaN,NaN,NaN,0.751796,0.09,0.59,0.73,0.9,2.48,0.23396
id,6427,6427,us7000than,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 10. Guardado del dataset procesado

Se guarda una copia procesada en `data/processed/` para que las siguientes etapas puedan reutilizarla sin ejecutar nuevamente toda la transformación. El archivo crudo de `data/raw/` no se modifica.

In [58]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)

## 11. Conclusión

Luego de aplicar las transformaciones definidas a partir de la EDA, se obtiene un conjunto de datos limpio y preparado para la siguiente etapa del proyecto. El dataset procesado queda disponible para continuar con el desarrollo del modelado.